In [4]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize

# Define the target function
def target_function(x):
    # Example target function: sum of 6 drivers
    return np.sum(x)

# Acquisition function: Expected Improvement (EI)
def acquisition_function(x, gp, y_max):
    x = np.array(x).reshape(1, -1)
    mean, std = gp.predict(x, return_std=True)
    std = std.reshape(-1, 1)
    z = (mean - y_max) / (std + 1e-9)
    ei = (mean - y_max) * norm.cdf(z) + std * norm.pdf(z)
    return -ei.ravel()  # Negative for minimization

# Bayesian Optimization
def bayesian_optimization(n_iter, bounds, gp):
    # Random initialization
    x_samples = np.random.uniform(bounds[:, 0], bounds[:, 1], size=(5, bounds.shape[0]))
    y_samples = np.array([target_function(x) for x in x_samples])

    for _ in range(n_iter):
        # Fit the Gaussian Process model
        gp.fit(x_samples, y_samples)

        # Find the next point to sample
        y_max = max(y_samples)
        res = minimize(
            fun=lambda x: acquisition_function(x, gp, y_max),
            x0=np.random.uniform(bounds[:, 0], bounds[:, 1], bounds.shape[0]),
            bounds=bounds,
            method="L-BFGS-B"
        )
        x_next = res.x

        # Evaluate the target function
        y_next = target_function(x_next)

        # Update samples
        x_samples = np.vstack((x_samples, x_next))
        y_samples = np.append(y_samples, y_next)

    # Return the best result
    best_idx = np.argmax(y_samples)
    return x_samples[best_idx], y_samples[best_idx]

if __name__ == "__main__":
    from scipy.stats import norm

    # Define the bounds for each driver (1 to 100 for 6 drivers)
    bounds = np.array([[1, 100]] * 6)

    # Define the Gaussian Process model
    kernel = Matern(nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)

    # Perform Bayesian Optimization
    n_iter = 100  # Number of iterations
    best_x, best_y = bayesian_optimization(n_iter, bounds, gp)

    print("Best input (drivers):", best_x)
    print("Maximum target value:", best_y)

Best input (drivers): [54.32845166 68.81174802 76.29841486 91.53330021 86.43457189 93.20535747]
Maximum target value: 470.6118441036638
